In [ ]:
!pip install timm -q

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import autocast, GradScaler
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler, Dataset
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score
from PIL import Image
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
# ── CHANGE THIS BETWEEN RUNS ──────────────────────────────────────
RANDOM_STATE = 0   # use 42, 0, 7, 123 for 4 separate runs
# ─────────────────────────────────────────────────────────────────

TRAIN_DIR = '/kaggle/input/competitions/cse-281-spring-26-scene-style-classification/StyleClassificationIndoors/StyleClassificationIndoors/train'
TEST_DIR  = '/kaggle/input/competitions/cse-281-spring-26-scene-style-classification/StyleClassificationIndoors/StyleClassificationIndoors/test'
SAVE_PATH = f'/kaggle/working/best_convnextv2b_rs{RANDOM_STATE}.pth'

print('Train exists:', os.path.exists(TRAIN_DIR))
print('Random state:', RANDOM_STATE)
print('Save path:   ', SAVE_PATH)

def remove_bad_images(root):
    removed = 0
    for folder, _, files in os.walk(root):
        for file in files:
            path = os.path.join(folder, file)
            try:
                with Image.open(path) as img:
                    img.verify()
            except Exception:
                os.remove(path)
                removed += 1
    print(f'Removed {removed} bad images')

remove_bad_images(TRAIN_DIR)

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2),
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

In [ ]:
full_train = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transforms)
full_val   = datasets.ImageFolder(root=TRAIN_DIR, transform=val_transforms)
print('Total images:', len(full_train))

targets = full_train.targets
indices = list(range(len(targets)))
train_idx, val_idx = train_test_split(
    indices, test_size=0.2, random_state=RANDOM_STATE, stratify=targets
)
print(f'Train: {len(train_idx)} | Val: {len(val_idx)}')

train_dataset = Subset(full_train, train_idx)
val_dataset   = Subset(full_val,   val_idx)

train_targets  = [targets[i] for i in train_idx]
class_counts   = np.bincount(train_targets)
class_weights  = 1.0 / class_counts
sample_weights = [class_weights[t] for t in train_targets]
sampler = WeightedRandomSampler(
    weights=torch.FloatTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False,
                          num_workers=2, pin_memory=True)

In [ ]:
# ============================================================
# CONVNEXT V2 ARCHITECTURE — Written from scratch
# Pipeline:
# Image → Stem → Stage 0 → Stage 1 → Stage 2 → Stage 3
# → Global Average Pool → LayerNorm → EnhancedHead → 17 logits
# ============================================================


# ── 1. GLOBAL RESPONSE NORMALIZATION ─────────────────────────────
# Key innovation in ConvNeXt V2 over V1.
# Creates competition between feature channels — amplifies
# strongly responding channels, suppresses weak ones.
# Prevents feature collapse where all channels learn the same thing.
class GRN(nn.Module):
    def __init__(self, dim):
        super().__init__()
        # Learnable scale and bias — initialized to zero
        # so GRN starts as identity and learns gradually
        self.gamma = nn.Parameter(torch.zeros(1, 1, 1, dim))
        self.beta  = nn.Parameter(torch.zeros(1, 1, 1, dim))

    def forward(self, x):
        # Compute L2 norm across spatial dimensions for each channel
        Gx = torch.norm(x, p=2, dim=(1, 2), keepdim=True)       # (B, 1, 1, C)
        # Normalize by mean norm — gives relative channel importance
        Nx = Gx / (Gx.mean(dim=-1, keepdim=True) + 1e-6)        # (B, 1, 1, C)
        # Apply learned scale/bias with residual connection
        return self.gamma * (x * Nx) + self.beta + x


# ── 2. STOCHASTIC DEPTH (DropPath) ───────────────────────────────
# Drops entire residual paths randomly during training.
# Stronger regularizer than dropout — forces the network
# to not rely on any single block for predictions.
# Drop probability increases linearly from 0 (early blocks)
# to drop_path_rate (last block) — deeper blocks dropped more.
class StochasticDepth(nn.Module):
    def __init__(self, drop_prob=0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if not self.training or self.drop_prob == 0.0:
            return x
        keep_prob   = 1 - self.drop_prob
        # One random binary value per sample in the batch
        shape       = (x.shape[0],) + (1,) * (x.ndim - 1)
        noise       = torch.rand(shape, dtype=x.dtype, device=x.device)
        noise       = torch.floor(noise + keep_prob)   # 0 or 1
        return x * noise / keep_prob                   # rescale to maintain expected value


# ── 3. LAYERNORM FOR NCHW TENSORS ────────────────────────────────
# Standard LayerNorm expects (B, N, C) but ConvNeXt uses (B, C, H, W).
# This wrapper permutes to NHWC, normalizes, then permutes back.
class LayerNorm2d(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.bias   = nn.Parameter(torch.zeros(dim))
        self.eps    = eps

    def forward(self, x):
        # x: (B, C, H, W) → permute → normalize → permute back
        return F.layer_norm(
            x.permute(0, 2, 3, 1),
            self.weight.shape, self.weight, self.bias, self.eps
        ).permute(0, 3, 1, 2)


# ── 4. CONVNEXT V2 BLOCK ─────────────────────────────────────────
# Core building block. Pipeline per block:
# input
#   → Depthwise Conv 7×7  (each channel processed separately — local features)
#   → LayerNorm           (normalize activations in NHWC format)
#   → Pointwise expand    (768→3072, channel mixing and expansion)
#   → GELU                (non-linearity)
#   → GRN                 (channel competition — V2 innovation)
#   → Pointwise compress  (3072→768, project back)
#   → StochasticDepth     (randomly drop entire block)
#   → + shortcut          (residual connection)
class ConvNeXtV2Block(nn.Module):
    def __init__(self, dim, drop_path=0.0):
        super().__init__()
        # Depthwise conv: groups=dim means each channel has its own 7x7 filter
        # Captures local spatial patterns within each feature channel
        self.dwconv  = nn.Conv2d(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm    = nn.LayerNorm(dim, eps=1e-6)         # normalize in NHWC
        self.pwconv1 = nn.Linear(dim, dim * 4)             # expand channels
        self.act     = nn.GELU()                           # GELU non-linearity
        self.grn     = GRN(dim * 4)                        # channel competition
        self.pwconv2 = nn.Linear(dim * 4, dim)             # compress back
        self.drop_path = StochasticDepth(drop_path)

    def forward(self, x):
        shortcut = x
        x = self.dwconv(x)                       # (B, C, H, W) spatial mixing
        x = x.permute(0, 2, 3, 1)               # NCHW → NHWC for LayerNorm
        x = self.norm(x)
        x = self.pwconv1(x)                      # channel expand
        x = self.act(x)
        x = self.grn(x)                          # GRN channel competition
        x = self.pwconv2(x)                      # channel compress
        x = x.permute(0, 3, 1, 2)               # NHWC → NCHW
        x = shortcut + self.drop_path(x)         # residual connection
        return x


# ── 5. FULL CONVNEXT V2 BASE ENCODER ─────────────────────────────
# ConvNeXt V2 Base configuration:
# depths = [3, 3, 27, 3]   — blocks per stage
# dims   = [128, 256, 512, 1024] — channels per stage
#
# Pipeline:
# Image → Stem → Stage0 → Down → Stage1 → Down → Stage2 → Down → Stage3
# → Global Average Pool → LayerNorm → 1024-dim feature vector
class ConvNeXtV2Base(nn.Module):
    def __init__(self, drop_path_rate=0.2):
        super().__init__()
        # ConvNeXt V2 Base architecture parameters
        depths = [3, 3, 27, 3]                  # blocks per stage
        dims   = [128, 256, 512, 1024]           # channels per stage
        total_blocks = sum(depths)               # 36 total blocks

        # Stochastic depth rates increase linearly from 0 to drop_path_rate
        dp_rates = [x.item() for x in
                    torch.linspace(0, drop_path_rate, total_blocks)]

        # ── Stem ──────────────────────────────────────────────────
        # Initial patch embedding: 4×4 non-overlapping conv
        # Converts (B, 3, 224, 224) → (B, 128, 56, 56)
        # Extracts low-level features: edges, colors, basic textures
        self.stem = nn.Sequential(
            nn.Conv2d(3, dims[0], kernel_size=4, stride=4),
            LayerNorm2d(dims[0])
        )

        # ── Downsampling layers between stages ────────────────────
        # Each halves spatial resolution and doubles channels
        # Stage 0→1: (B, 128, 56, 56) → (B, 256, 28, 28)
        # Stage 1→2: (B, 256, 28, 28) → (B, 512, 14, 14)
        # Stage 2→3: (B, 512, 14, 14) → (B, 1024, 7, 7)
        self.downsample_layers = nn.ModuleList([
            nn.Sequential(
                LayerNorm2d(dims[i]),
                nn.Conv2d(dims[i], dims[i+1], kernel_size=2, stride=2)
            ) for i in range(3)
        ])

        # ── 4 Stages of ConvNeXt V2 Blocks ───────────────────────
        # Stage 0: 3 blocks, 128 channels, 56×56 spatial
        # Stage 1: 3 blocks, 256 channels, 28×28 spatial
        # Stage 2: 27 blocks, 512 channels, 14×14 spatial (main computation)
        # Stage 3: 3 blocks, 1024 channels, 7×7 spatial
        self.stages = nn.ModuleList()
        cur = 0
        for i in range(4):
            stage = nn.Sequential(*[
                ConvNeXtV2Block(dims[i], dp_rates[cur + j])
                for j in range(depths[i])
            ])
            self.stages.append(stage)
            cur += depths[i]

        # ── Final normalization ───────────────────────────────────
        # Applied after global average pooling
        self.norm = nn.LayerNorm(dims[-1], eps=1e-6)   # normalizes 1024-dim vector

    def forward(self, x):
        # Stem: extract initial patch features
        x = self.stem(x)                              # (B, 128, 56, 56)

        # Four stages with downsampling between them
        for i in range(4):
            x = self.stages[i](x)                     # process blocks
            if i < 3:
                x = self.downsample_layers[i](x)      # halve spatial resolution

        # (B, 1024, 7, 7) after all stages
        x = x.mean([-2, -1])                          # global average pool → (B, 1024)
        x = self.norm(x)                              # normalize feature vector
        return x                                       # 1024-dim image representation


# ── 6. ENHANCED CLASSIFICATION HEAD ──────────────────────────────
# Attaches to the 1024-dim backbone output.
# Uses a Cosine Classifier instead of standard linear:
# — normalizes both features and class weight vectors
# — measures cosine similarity (angle) rather than dot product
# — better for fine-grained tasks where classes are visually similar
# — scale=16 amplifies the cosine scores for stable cross-entropy
#
# Pipeline:
# 1024-dim → Linear(512) → BN → GELU → Dropout(0.3)
#          → Linear(512) → BN → GELU → Dropout(0.2)
#          → + residual projection
#          → L2 normalize → cosine similarity × scale → 17 logits
class EnhancedHead(nn.Module):
    def __init__(self, in_features, num_classes=17, scale=16.0):
        super().__init__()
        # Two-layer bottleneck with BatchNorm
        self.fc1      = nn.Linear(in_features, 512)
        self.bn1      = nn.BatchNorm1d(512)
        self.act1     = nn.GELU()
        self.drop1    = nn.Dropout(0.3)
        self.fc2      = nn.Linear(512, 512)
        self.bn2      = nn.BatchNorm1d(512)
        self.act2     = nn.GELU()
        self.drop2    = nn.Dropout(0.2)
        # Residual shortcut from input to bottleneck output
        self.residual = nn.Linear(in_features, 512)
        # Cosine classifier: learnable class prototype vectors
        self.scale    = scale
        self.weight   = nn.Parameter(torch.randn(num_classes, 512))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x):
        res = self.residual(x)                      # shortcut path
        out = self.drop1(self.act1(self.bn1(self.fc1(x))))
        out = self.drop2(self.act2(self.bn2(self.fc2(out))))
        out = out + res                             # add residual
        out = F.normalize(out, dim=1)               # L2 normalize features
        w   = F.normalize(self.weight, dim=1)       # L2 normalize class prototypes
        return self.scale * (out @ w.T)             # scaled cosine similarity


# ── 7. FULL MODEL — BACKBONE + HEAD ──────────────────────────────
class ConvNeXtV2Classifier(nn.Module):
    def __init__(self, num_classes=17, drop_path_rate=0.2):
        super().__init__()
        self.backbone = ConvNeXtV2Base(drop_path_rate=drop_path_rate)
        self.head     = EnhancedHead(in_features=1024, num_classes=num_classes)

    def forward(self, x):
        features = self.backbone(x)    # (B, 1024)
        return self.head(features)     # (B, 17)

    def load_pretrained_weights(self):
        """
        Loads ConvNeXt V2 Base pretrained weights from timm.
        Only backbone weights are loaded — EnhancedHead is always
        trained from scratch since it is a custom architecture.
        """
        print('Loading ConvNeXt V2 Base pretrained weights from timm...')
        pretrained      = timm.create_model('convnextv2_base', pretrained=True)
        pretrained_dict = pretrained.state_dict()
        our_dict        = self.state_dict()

        # Map timm weight names to our architecture names
        name_map = {}
        for k in pretrained_dict.keys():
            new_k = k
            new_k = new_k.replace('stem.0.',              'backbone.stem.0.')
            new_k = new_k.replace('stem.1.',              'backbone.stem.1.')
            new_k = new_k.replace('stages.',              'backbone.stages.')
            new_k = new_k.replace('downsample_layers.',   'backbone.downsample_layers.')
            new_k = new_k.replace('norm.',                'backbone.norm.')
            # ConvNeXt V2 block internals
            new_k = new_k.replace('.gamma',               '.grn.gamma')
            new_k = new_k.replace('.beta',                '.grn.beta')
            name_map[k] = new_k

        matched, skipped = 0, 0
        for old_k, new_k in name_map.items():
            if new_k in our_dict and pretrained_dict[old_k].shape == our_dict[new_k].shape:
                our_dict[new_k] = pretrained_dict[old_k]
                matched += 1
            else:
                skipped += 1

        self.load_state_dict(our_dict, strict=False)
        print(f'Loaded {matched} layers | Skipped {skipped} layers')
        print('EnhancedHead randomly initialized — will be trained from scratch.')
        del pretrained


# ── Build model and load pretrained weights ───────────────────────
model = ConvNeXtV2Classifier(num_classes=17, drop_path_rate=0.2)
model.load_pretrained_weights()

for param in model.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,}')

model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.2)

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        if np.random.rand() < 0.5:
            lam   = np.random.beta(0.4, 0.4)
            index = torch.randperm(images.size(0), device=device)
            images = lam * images + (1 - lam) * images[index]
            with autocast('cuda'):
                out  = model(images)
                loss = lam * criterion(out, labels) + (1 - lam) * criterion(out, labels[index])
        else:
            with autocast('cuda'):
                out  = model(images)
                loss = criterion(out, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        all_preds.extend(out.argmax(dim=1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), balanced_accuracy_score(all_labels, all_preds)

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            with autocast('cuda'):
                out  = model(images)
                loss = criterion(out, labels)
            total_loss += loss.item()
            all_preds.extend(out.argmax(dim=1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader), balanced_accuracy_score(all_labels, all_preds)

scaler = GradScaler('cuda')

optimizer = AdamW([
    {'params': [p for n, p in model.named_parameters()
                if 'head' not in n], 'lr': 1e-5},
    {'params': model.head.parameters(), 'lr': 1e-4},
], weight_decay=0.1)

NUM_EPOCHS    = 40
WARMUP_EPOCHS = 3

def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / (NUM_EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + __import__('math').cos(__import__('math').pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

layers_to_unfreeze = [
    [model.backbone.stages[3]],
    [model.backbone.stages[2]],
    [model.backbone.stages[1]],
    [model.backbone.stages[0]],
    [model.backbone.stem],
]

patience        = 5
plateau_counter = 0
unfreeze_index  = 0
best_val_acc    = 0
best_val_loss   = float('inf')

print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Train BAcc':>10} | {'Val Loss':>8} | {'Val BAcc':>8} | {'Status':>20}")
print('-' * 75)

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, scaler, device)
    val_loss,   val_acc   = validate(model, val_loader, criterion, device)
    scheduler.step()

    is_best = False
    if val_acc > best_val_acc:
        best_val_acc  = val_acc
        best_val_loss = val_loss
        is_best = True
    elif val_acc == best_val_acc and val_loss < best_val_loss:
        best_val_loss = val_loss
        is_best = True

    if is_best:
        plateau_counter = 0
        torch.save(model.state_dict(), SAVE_PATH)
        print(f'  Checkpoint saved: {os.path.getsize(SAVE_PATH) / 1e6:.1f} MB')
    else:
        plateau_counter += 1

    status = '✓ New Best' if is_best else f'Plateau: {plateau_counter}/{patience}'
    print(f"{epoch+1:>5} | {train_loss:>10.4f} | {train_acc:>10.4f} | "
          f"{val_loss:>8.4f} | {val_acc:>8.4f} | {status:>20}")

    if plateau_counter >= patience and unfreeze_index < len(layers_to_unfreeze):
        print(f"\n  🔥 Plateau hit! Unfreezing stage group {unfreeze_index + 1}...")
        for module in layers_to_unfreeze[unfreeze_index]:
            for param in module.parameters():
                param.requires_grad = True
        unfreeze_index  += 1
        plateau_counter  = 0

        print('  → Resetting AdamW with differential learning rates')
        optimizer = AdamW([
            {'params': model.head.parameters(), 'lr': 1e-4},
            {'params': [p for n, p in model.named_parameters()
                        if 'head' not in n and p.requires_grad], 'lr': 5e-6},
        ], weight_decay=0.1)

        remaining_epochs = NUM_EPOCHS - epoch - 1
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max(remaining_epochs, 1), eta_min=1e-7
        )

print(f'\nBest Val Balanced Accuracy: {best_val_acc:.4f}')

from IPython.display import FileLink, display
display(FileLink(SAVE_PATH))

In [ ]:
# ── Ensemble + TTA Submission ─────────────────────────────────────

class TestDataset(Dataset):
    def __init__(self, folder_path, transform=None):
        self.folder_path = folder_path
        self.image_files = [f for f in os.listdir(folder_path)
                            if f.endswith(('.jpg', '.jpeg', '.png'))]
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.folder_path, img_name)
        try:
            image = Image.open(img_path).convert('RGB')
        except Exception:
            image = Image.new('RGB', (224, 224), (0, 0, 0))
        if self.transform:
            image = self.transform(image)
        return image, img_name

tta_transforms = [
    transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]),
    transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]),
    transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]),
]

checkpoint_paths = [
    '/kaggle/working/best_convnextv2b_rs42.pth',
    '/kaggle/working/best_convnextv2b_rs0.pth',
    '/kaggle/working/best_convnextv2b_rs7.pth',
    '/kaggle/working/best_convnextv2b_rs123.pth',
]
checkpoint_paths = [p for p in checkpoint_paths if os.path.exists(p)]
print(f'Ensembling {len(checkpoint_paths)} models with {len(tta_transforms)} TTA transforms each')

all_probs   = None
image_names = None

for ckpt_path in checkpoint_paths:
    print(f'Loading {ckpt_path}...')
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model.eval()

    for tfm_idx, tfm in enumerate(tta_transforms):
        test_dataset = TestDataset(folder_path=TEST_DIR, transform=tfm)
        test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

        probs = []
        names = []

        with torch.no_grad():
            for i, (images, batch_names) in enumerate(test_loader):
                images = images.to(device)
                with autocast('cuda'):
                    out = model(images)
                probs.extend(torch.softmax(out, dim=1).cpu().numpy())
                names.extend(batch_names)
                if (i + 1) % 25 == 0:
                    print(f'  TTA {tfm_idx+1}/{len(tta_transforms)} | Batch {i+1}/{len(test_loader)}')

        probs = np.array(probs)
        if all_probs is None:
            all_probs   = probs
            image_names = names
        else:
            all_probs += probs

final_preds = all_probs.argmax(axis=1)

submission_df = pd.DataFrame({'ImageName': image_names, 'ClassLabel': final_preds})
submission_df['sort_number'] = submission_df['ImageName'].str.extract(r'(\d+)').astype(int)
submission_df = submission_df.sort_values(by='sort_number')
submission_df = submission_df.drop(columns=['sort_number'])

submission_df.to_csv('/kaggle/working/submission_ensemble.csv', index=False)
print(f'SUCCESS! Saved submission_ensemble.csv with {len(submission_df)} predictions in numerical order.')
print(submission_df['ClassLabel'].value_counts().sort_index())